# Claims Bronze-to-Silver Transformation

## Purpose

Transform the raw Kaggle claims dataset from the Bronze layer into a
clean, typed, standardized Silver Delta table.

### Source
`health_insurance.bronze.claims_raw`

### Target
`health_insurance.silver.claims`

### Responsibilities

- Standardize column names
- Cast source fields to appropriate data types
- Standardize categorical values
- Add useful derived attributes
- Preserve source lineage metadata

Formal data-quality rule enforcement is implemented separately in the
dedicated data-quality stage.



In [0]:
# ============================================================
# Project configuration
# ============================================================

CATALOG = "health_insurance"

SOURCE_TABLE = f"{CATALOG}.bronze.claims_raw"
TARGET_TABLE = f"{CATALOG}.silver.claims"

print("Source:", SOURCE_TABLE)
print("Target:", TARGET_TABLE)

In [0]:
# ============================================================
# Loading Bronze claims
# ============================================================

claims_bronze_df = spark.table(SOURCE_TABLE)

print(
    f"Rows: {claims_bronze_df.count():,}"
)

print(
    f"Columns: {len(claims_bronze_df.columns)}"
)

display(claims_bronze_df.limit(5))

In [0]:
# ============================================================
# Inspecting Bronze schema
# ============================================================

claims_bronze_df.printSchema()

In [0]:
# ============================================================
# Standardizing column naming
# ============================================================

claims_standardized_df = (
    claims_bronze_df

    .withColumnRenamed("Patient_ID", "patient_id")
    .withColumnRenamed("Policy_Number", "policy_number")
    .withColumnRenamed("Claim_ID", "claim_id")
    .withColumnRenamed("Claim_Date", "claim_date")
    .withColumnRenamed("Service_Date", "service_date")
    .withColumnRenamed(
        "Policy_Expiration_Date",
        "policy_expiration_date"
    )
    .withColumnRenamed("Claim_Amount", "claim_amount")
    .withColumnRenamed("Patient_Age", "patient_age")
    .withColumnRenamed("Patient_Gender", "patient_gender")
    .withColumnRenamed("Patient_City", "patient_city")
    .withColumnRenamed("Patient_State", "patient_state")
    .withColumnRenamed("Hospital_ID", "hospital_id")
    .withColumnRenamed("Provider_Type", "provider_type")
    .withColumnRenamed(
        "Provider_Specialty",
        "provider_specialty"
    )
    .withColumnRenamed("Provider_City", "provider_city")
    .withColumnRenamed("Provider_State", "provider_state")
    .withColumnRenamed("Diagnosis_Code", "diagnosis_code")
    .withColumnRenamed("Procedure_Code", "procedure_code")
    .withColumnRenamed(
        "Number_of_Procedures",
        "number_of_procedures"
    )
    .withColumnRenamed("Admission_Type", "admission_type")
    .withColumnRenamed("Discharge_Type", "discharge_type")
    .withColumnRenamed(
        "Length_of_Stay_Days",
        "length_of_stay_days"
    )
    .withColumnRenamed("Service_Type", "service_type")
    .withColumnRenamed(
        "Deductible_Amount",
        "deductible_amount"
    )
    .withColumnRenamed("CoPay_Amount", "copay_amount")
    .withColumnRenamed(
        "Number_of_Previous_Claims_Patient",
        "previous_claims_patient"
    )
    .withColumnRenamed(
        "Number_of_Previous_Claims_Provider",
        "previous_claims_provider"
    )
    .withColumnRenamed(
        "Provider_Patient_Distance_Miles",
        "provider_patient_distance_miles"
    )
    .withColumnRenamed(
        "Claim_Submitted_Late",
        "claim_submitted_late"
    )
    .withColumnRenamed(
        "Is_Fraudulent",
        "is_fraudulent"
    )
)

In [0]:
# ============================================================
# Applying explicit Silver data types
# ============================================================

from pyspark.sql import functions as F

claims_typed_df = (
    claims_standardized_df

    .withColumn(
        "patient_id",
        F.col("patient_id").cast("long")
    )

    .withColumn(
        "claim_id",
        F.col("claim_id").cast("long")
    )

    .withColumn(
        "hospital_id",
        F.col("hospital_id").cast("long")
    )

    .withColumn(
        "claim_date",
        F.to_date("claim_date")
    )

    .withColumn(
        "service_date",
        F.to_date("service_date")
    )

    .withColumn(
        "policy_expiration_date",
        F.to_date("policy_expiration_date")
    )

    .withColumn(
        "claim_amount",
        F.col("claim_amount").cast("decimal(18,2)")
    )

    .withColumn(
        "deductible_amount",
        F.col("deductible_amount").cast("decimal(18,2)")
    )

    .withColumn(
        "copay_amount",
        F.col("copay_amount").cast("decimal(18,2)")
    )

    .withColumn(
        "provider_patient_distance_miles",
        F.col("provider_patient_distance_miles").cast("double")
    )

    .withColumn(
        "patient_age",
        F.col("patient_age").cast("int")
    )

    .withColumn(
        "number_of_procedures",
        F.col("number_of_procedures").cast("int")
    )

    .withColumn(
        "length_of_stay_days",
        F.col("length_of_stay_days").cast("int")
    )

    .withColumn(
        "previous_claims_patient",
        F.col("previous_claims_patient").cast("int")
    )

    .withColumn(
        "previous_claims_provider",
        F.col("previous_claims_provider").cast("int")
    )

    .withColumn(
        "claim_submitted_late",
        F.col("claim_submitted_late").cast("boolean")
    )

    .withColumn(
        "is_fraudulent",
        F.col("is_fraudulent").cast("boolean")
    )
)

In [0]:
# ============================================================
# Standardizing categorical text
# ============================================================

claims_clean_df = (
    claims_typed_df

    .withColumn(
        "patient_gender",
        F.upper(F.trim("patient_gender"))
    )

    .withColumn(
        "patient_city",
        F.initcap(F.trim("patient_city"))
    )

    .withColumn(
        "patient_state",
        F.upper(F.trim("patient_state"))
    )

    .withColumn(
        "provider_type",
        F.upper(F.trim("provider_type"))
    )

    .withColumn(
        "provider_specialty",
        F.initcap(F.trim("provider_specialty"))
    )

    .withColumn(
        "provider_city",
        F.initcap(F.trim("provider_city"))
    )

    .withColumn(
        "provider_state",
        F.upper(F.trim("provider_state"))
    )

    .withColumn(
        "admission_type",
        F.upper(F.trim("admission_type"))
    )

    .withColumn(
        "discharge_type",
        F.upper(F.trim("discharge_type"))
    )

    .withColumn(
        "service_type",
        F.upper(F.trim("service_type"))
    )
)

In [0]:
# calculating the claims submission delay and adding it to the dataframe( claim_date - service_date)

claims_enriched_df = (
    claims_clean_df

    .withColumn(
        "claim_submission_delay_days",
        F.datediff(
            F.col("claim_date"),
            F.col("service_date")
        )
    )
)

In [0]:
# defining the claim_amount band c based on the claim_amount_band column to be adjusted later after noting the data statistics

claims_enriched_df = (
    claims_enriched_df

    .withColumn(
        "claim_amount_band",
        F.when(
            F.col("claim_amount") < 1000,
            "LOW"
        )
        .when(
            F.col("claim_amount") < 5000,
            "MEDIUM"
        )
        .when(
            F.col("claim_amount") < 10000,
            "HIGH"
        )
        .otherwise("VERY_HIGH")
    )
)

In [0]:
# adding a column todefine age groups

claims_enriched_df = (
    claims_enriched_df

    .withColumn(
        "patient_age_group",
        F.when(F.col("patient_age") < 18, "UNDER_18")
         .when(F.col("patient_age") < 35, "18_34")
         .when(F.col("patient_age") < 50, "35_49")
         .when(F.col("patient_age") < 65, "50_64")
         .otherwise("65_PLUS")
    )
)

In [0]:
#adding a transformed at timestamp to preserve lineage metadata

claims_silver_df = (
    claims_enriched_df
    .withColumn(
        "_silver_transformed_at",
        F.current_timestamp()
    )
)

In [0]:
# ============================================================
# Inspecting transformed claims
# ============================================================

claims_silver_df.printSchema()

display(
    claims_silver_df.select(
        "claim_id",
        "patient_id",
        "claim_date",
        "service_date",
        "claim_amount",
        "claim_amount_band",
        "patient_age",
        "patient_age_group",
        "claim_submission_delay_days"
    ).limit(20)
)

In [0]:
# comparing the number of rows in the bronze and silver tables before writing to table

bronze_count = claims_bronze_df.count()
silver_count = claims_silver_df.count()

print(f"Bronze rows: {bronze_count:,}")
print(f"Silver rows: {silver_count:,}")
print(f"Difference: {bronze_count - silver_count:,}")

In [0]:
# ============================================================
# Persisting transformed claims to Silver
# ============================================================

(
    claims_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(TARGET_TABLE)
)

print(
    f"Created Silver table: {TARGET_TABLE}"
)

In [0]:
%sql
-- queiring the table for verification 

SELECT
    claim_amount_band,
    COUNT(*) AS claims,
    ROUND(AVG(claim_amount), 2) AS avg_claim_amount
FROM health_insurance.silver.claims
GROUP BY claim_amount_band
ORDER BY avg_claim_amount;

## Transformation Result

The Kaggle health insurance claims data was successfully transformed
from the Bronze layer into a standardized Silver Delta table.

### Source

`health_insurance.bronze.claims_raw`

### Target

`health_insurance.silver.claims`

### Transformations Applied

- Standardized source column names using snake_case naming conventions.
- Applied explicit data types to identifiers, dates, numeric values,
  monetary values, and boolean fields.
- Standardized categorical and text values.
- Converted source date fields into proper Spark date types.
- Created `claim_submission_delay_days` from the service and claim dates.
- Created `claim_amount_band` for claim-value categorization.
- Created `patient_age_group` for demographic analysis.
- Preserved Bronze ingestion metadata for lineage and traceability.
- Added `_silver_transformed_at` to record Silver processing time.

### Row Reconciliation

The Bronze and Silver row counts were compared before persistence.

No records were intentionally removed during this transformation stage.
The Silver transformation therefore preserves the complete Bronze
claims dataset while improving its structure and usability.

### Data Quality Boundary

This notebook performs cleansing, standardization, and schema
transformation only.

Formal data-quality enforcement is intentionally handled separately
in the project's dedicated data-quality stage.

Examples of rules that will be implemented later include:

- Required claim and patient identifiers
- Valid claim amounts
- Valid patient ages
- Logical service and claim dates
- Duplicate claim detection
- Valid categorical values
- Quarantine, drop, warning, and pipeline-failure behavior

### Architecture

Kaggle CSV  
↓  
`health_insurance.bronze.claims_raw`  
↓  
**Bronze-to-Silver transformation**  
↓  
`health_insurance.silver.claims`  
↓  
Data Quality  
↓  
Gold analytical model

